Data Generator

In [5]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 1000

# Basic customer information
df = pd.DataFrame({
    "customer_id": range(1, n + 1),

    "age": np.random.randint(18, 70, n),

    "monthly_spend": np.random.randint(20, 300, n),

    "support_calls": np.random.randint(0, 10, n),

    "tenure_months": np.random.randint(1, 72, n),

    "contract_type": np.random.choice(
        ["Monthly", "Yearly", "Two-year"],
        n,
        p=[0.5, 0.3, 0.2]
    )
})


# Create churn logic
# Higher risk:
# - Monthly contract
# - More support calls
# - Low tenure
# - High spend

churn_probability = (
    0.05
    + (df["contract_type"] == "Monthly") * 0.25
    + (df["support_calls"] > 5) * 0.20
    + (df["tenure_months"] < 12) * 0.20
    + (df["monthly_spend"] > 200) * 0.10
)


# Add randomness
churn_probability = np.clip(
    churn_probability,
    0,
    0.9
)


df["churn"] = (
    np.random.random(n) < churn_probability
).astype(int)


# Add missing values intentionally

for col in [
    "age",
    "monthly_spend",
    "contract_type"
]:
    missing_indexes = np.random.choice(
        df.index,
        size=30,
        replace=False
    )

    df.loc[missing_indexes, col] = np.nan


# Save CSV

df.to_csv(
    "customer_churn.csv",
    index=False
)


print(df.head())
print(df["churn"].value_counts())

   customer_id   age  monthly_spend  support_calls  tenure_months  \
0            1  56.0          246.0              1             25   
1            2  69.0           34.0              2             23   
2            3  46.0          211.0              9             24   
3            4  32.0          138.0              5             38   
4            5  60.0           47.0              6             28   

  contract_type  churn  
0       Monthly      1  
1      Two-year      0  
2       Monthly      1  
3        Yearly      0  
4       Monthly      1  
churn
0    700
1    300
Name: count, dtype: int64


now Lets Test the models

In [6]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import ( confusion_matrix,accuracy_score, classification_report, precision_score, f1_score, recall_score )

In [7]:
df = pd.read_csv("customer_churn.csv")

In [8]:
df.head()

,customer_id,age,monthly_spend,support_calls,tenure_months,contract_type,churn
0,1,56.0,246.0,1,25,Monthly,1
1,2,69.0,34.0,2,23,Two-year,0
2,3,46.0,211.0,9,24,Monthly,1
3,4,32.0,138.0,5,38,Yearly,0
4,5,60.0,47.0,6,28,Monthly,1


In [9]:
x = df[["age", "monthly_spend", "support_calls", "tenure_months", "contract_type"]]
y = df["churn"]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(x, y,stratify=y , test_size=0.2, random_state=42)

In [11]:
numerical_feature = ["age",	"monthly_spend",	"support_calls",	"tenure_months"]
categorical_feature = ["contract_type"]

In [12]:
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [13]:
categorail_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder())
])

In [14]:
preprocessor = ColumnTransformer([
    ('numerical', numerical_pipeline, numerical_feature),
    ('categorical', categorail_pipeline, categorical_feature)
])

In [15]:
logistic_pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('model',LogisticRegression())
])

In [16]:
tree_pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('model',DecisionTreeClassifier(    max_depth=5,
    min_samples_split=20,class_weight="balanced",
random_state=42))
])


In [17]:
forest_pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('model',RandomForestClassifier(    max_depth=5,
    min_samples_split=20,class_weight="balanced",random_state=42))
])


In [18]:
logistic_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'monthly_spend',
                                                   'support_calls',
                                                   'tenure_months']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder())]),
                                                  ['contract_type'])])),
                ('model', LogisticRegression())])

In [19]:
tree_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'monthly_spend',
                                                   'support_calls',
                                                   'tenure_months']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder())]),
                                                  ['contract_type'])])),
                ('model',
                 DecisionTreeClassifier(class_weight='balanced', max_depth=5,
                                        min_samples_split=20,
                                        random_state=42))])

In [20]:
forest_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'monthly_spend',
                                                   'support_calls',
                                                   'tenure_months']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder())]),
                                                  ['contract_type'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced', max_depth=5,
                                        min_samples_split=20,
                                        random_state=42))])

In [21]:
logistic_prob = logistic_pipeline.predict_proba(X_test)[:,1]

tree_prob = tree_pipeline.predict_proba(X_test)[:,1]

forest_prob = forest_pipeline.predict_proba(X_test)[:,1]

In [22]:
threshold = 0.3
logistic_prob_03 = (logistic_prob >= threshold).astype(int)
tree_prob_03 = (tree_prob >= threshold).astype(int)
forest_prob_03 = (forest_prob >= threshold).astype(int)

In [23]:
def evaluate_model(name, y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    print(classification_report(y_test, y_pred))
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    }

In [24]:
results = []
results.append(
    evaluate_model("Logistic Regression", y_test, logistic_prob_03)
)
results.append(
    evaluate_model("Decision Tree", y_test, tree_prob_03)
)
results.append(
    evaluate_model("Random Forest", y_test, forest_prob_03)
)

[[90 50]
 [25 35]]
              precision    recall  f1-score   support

           0       0.78      0.64      0.71       140
           1       0.41      0.58      0.48        60

    accuracy                           0.62       200
   macro avg       0.60      0.61      0.59       200
weighted avg       0.67      0.62      0.64       200

[[58 82]
 [10 50]]
              precision    recall  f1-score   support

           0       0.85      0.41      0.56       140
           1       0.38      0.83      0.52        60

    accuracy                           0.54       200
   macro avg       0.62      0.62      0.54       200
weighted avg       0.71      0.54      0.55       200

[[45 95]
 [ 4 56]]
              precision    recall  f1-score   support

           0       0.92      0.32      0.48       140
           1       0.37      0.93      0.53        60

    accuracy                           0.51       200
   macro avg       0.64      0.63      0.50       200
weighted avg     

In [25]:
result_df = pd.DataFrame(results)
result_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.625,0.411765,0.583333,0.482759
1,Decision Tree,0.540,0.378788,0.833333,0.520833
2,Random Forest,0.505,0.370861,0.933333,0.530806
